In [1]:
import qutip as qt
from qutip import tensor, basis
import numpy as np
import matplotlib.pyplot as plt
from quantum_logical.channel import AmplitudeDamping, PhaseDamping
from quantum_logical.trotter import TrotterGroup
from tqdm import tqdm
from quantum_logical.operators import selective_destroy
from scipy.optimize import curve_fit

In [2]:
# generating parameters and creating initial state
T1 = 50
T2 = 25
N = 5
dim = 3
trotter_dt = .01
amp_damp_channel = AmplitudeDamping(T1, num_qubits=N, hilbert_space_dim=dim)
phase_damp_channel = PhaseDamping(T1, T2, num_qubits=N, hilbert_space_dim=dim)
trotter = TrotterGroup(
    continuous_operators=[amp_damp_channel, phase_damp_channel],
    trotter_dt=trotter_dt,
)


The qutrit subspace of | g $\rangle$ and | f $\rangle$ $\newline$
The qutrit error state is the | e $\rangle$

In [9]:
# creating the initial state of the system 
psi0 = tensor(basis(dim, 2), basis(dim, 0), basis(dim, 0), basis(dim, 0), basis(dim, 0))
rho0 = psi0 * psi0.dag()

Quantum object: dims = [[3, 3], [3, 3]], shape = (9, 9), type = oper, isherm = True
Qobj data =
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]

In [4]:
# creating the equivalent two qubit gates
# encoding set 
cnot1 = qt.cnot(N=5, target=1, control=0)
cnot2 = qt.cnot(N=5, target=2, control=0)

hadamard = (1 / np.sqrt(2)) * qt.Qobj([[1, 0, 1], [0, np.sqrt(2), 0], [1, 0, -1]])  # this has been put in the qutrit basis 
hadamard

# stabilizer set
cnot3 = qt.cnot(N=5, target=3, control=0)
cnot4 = qt.cnot(N=5, target=3, control=1)

cnot5 = qt.cnot(N=5, target=4, control=1)
cnot6 = qt.cnot(N=5, target=4, control=2)

# correction set 
# creating the z gates


C:\Users\girgi\AppData\Local\Temp\ipykernel_24244\439573999.py:2: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  cnot1 = qt.cnot(N=5, target=1, control=0)
C:\Users\girgi\AppData\Local\Temp\ipykernel_24244\439573999.py:3: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  cnot2 = qt.cnot(N=5, target=2, control=0)


Quantum object: dims = [[3], [3]], shape = (3, 3), type = oper, isherm = True
Qobj data =
[[ 0.70710678  0.          0.70710678]
 [ 0.          1.          0.        ]
 [ 0.70710678  0.         -0.70710678]]

Expanding gates into a larger dimensionality is easy $\newline$
To create the change matrix build an identity in the subspace of the basis you want to go from to the one you want to go to $\newline$
The important thing here is that the matrix should be of dimensionality m x n where m is the dimensionality you are moving from to the one you are going to $\newline$
Once this is done then you multiply these out and then add in the a matrix of the added dimension vector $\newline$
INSERT AN IMAGE FOR COMPREHENSION

In [54]:
# matrix_multiplier = np.zeros((2 * 2, 2 * 3)) # building the change matrix
# for k in range(2):
#     for i in range(2):
#         for j in range(3):
#             if i != [2, ]:
#                 matrix_multiplier[k * i + i, k * j + j] = 1
#                 print({k}, {i}, {j})
# qt.Qobj(matrix_multiplier)

# # build the array of elements that will get populated instead of building the matrix first
# Is = []
# js = []
# for i in range(2 * 2):
#     for j in range(2 * 3):
#         if i < 2:
#             Is.append(i)
#             js.append(j)
        


{0} {0} {0}
{0} {1} {1}
{1} {0} {0}
{1} {1} {1}


Quantum object: dims = [[4], [6]], shape = (4, 6), type = oper, isherm = False
Qobj data =
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]

: 

In [15]:
# create a general algorithm for converting qubit gates to qutrit gates
# maybe at some point add in the functionality to move around the workings of the gate this should not be difficult
def qubit_to_higher_dimension(qubit_gate, add_levels, choice, N):
    if choice == "add":
        # build the matrix based on the added levels
        matrix_multiplier = []
        for i in range(add_levels + 2):




Quantum object: dims = [[2], [3]], shape = (2, 3), type = oper, isherm = False
Qobj data =
[[0.3660254 0.        0.3660254]
 [0.        0.3660254 0.3660254]]